<a href="https://colab.research.google.com/github/pathilink/adyen_payment_optimization_case/blob/main/notebooks/05_hypothesis_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='#0ABF56'> Optimisation Data Analyst Case Study </font>

## <font color='#0ABF56'> 05 - Hypothesis Testing </font>

# Libraries

In [1]:
import pandas as pd
import numpy as np
import datetime
import seaborn as sns
from matplotlib import pyplot as plt

from statsmodels.stats.proportion import proportions_ztest

# Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df = pd.read_csv('/content/drive/MyDrive/test/adyen/data/processed/adyen_transactions_analysis.csv')

df.head()

,psp_reference,bin,scheme,issuername,shopper_interaction,avs_data_supplied,cvc_data_supplied,amount,raw_acquirer_response,creation_date,authorization,issuer_known,amount_range
0,1,400178.0,visa,BANCO DO BRASIL S.A.,Ecommerce,False,False,1.00,05 : Do not honor / A201 : 3D Secure Mandated,2019-06-01 00:19:00,False,True,Until 4.16
1,2,486348.0,visa,FIRST ATLANTIC BANK LIMITED,Ecommerce,False,False,6.48,00 : Approved or completed successfully,2019-06-01 00:22:00,True,True,4.16 to 19.55
2,3,482481.0,visa,ITAU UNIBANCO S.A.,Ecommerce,False,True,4.00,06 : Error,2019-06-01 00:46:00,False,True,Until 4.16
3,4,439267.0,visa,CAIXA ECONOMICA FEDERAL,Ecommerce,False,True,2.76,05 : Do not honor / A201 : 3D Secure Mandated,2019-06-01 01:02:00,False,True,Until 4.16
4,5,489347.0,visa,VTB BANK PJSC,Ecommerce,False,True,97.00,00 : Approved or completed successfully,2019-06-01 01:30:00,True,True,above 52.10


# Function

In [4]:
def run_proportion_test(success_a, size_a, success_b, size_b, alternative='two-sided'):

    successes = np.array([success_a, success_b])
    sizes = np.array([size_a, size_b])

    stat, pvalue = proportions_ztest(successes, sizes, alternative=alternative)

    return stat, pvalue

# Business context

The goal is to improve authorization rate by identifying actionable optimization opportunities across issuers, shopper interaction behavior, and transaction configuration.

<br>

**Metric**

$\text{AuthorizationRate} = \frac{\text{ApprovedTransactions}}{\text{TotalTransactions}}$

<br>

**Assumptions validation**

* Independence: transactions are assumed to be independent observations.

* Sample size adequacy: both groups contain sufficiently large samples to satisfy the normal approximation conditions required for the z-test.

## Smart CVC

**Business problem**

Some issuers (e.g. Nubank) are rejecting more transactions without a CVC due to anti-fraud policies.

<br>

**Analytical question**

Do transactions with CVC supplied have higher authorization rates than transactions without CVC supplied for Nubank-issued cards?

<br>

**Hypothesis**

H0: Transactions with CVC supplied have the same authorization rate as transactions without CVC supplied.
* $H_0: p_{CVC} = p_{NoCVC}$

H1: Transactions with CVC supplied have a higher authorization rate.
* $H_1: p_{CVC} > p_{NoCVC}$

<br>

**Comparison**

| Group     | Definition                 |
| :-------- | :------------------------- |
| Control   | Transactions without a CVC |
| Treatment | Transactions with a CVC    |


<br>

**Interpretation**

If p-value < 0.05:

* reject H0;
* statistical evidence that including CVC improves approval rates.


In [5]:
# only nubank
df_nubank = df[
    df['issuername'] == 'NU PAGAMENTOS SA'
]

In [6]:
# create group
group_cvc = df_nubank[df_nubank['cvc_data_supplied'] == True]
group_no_cvc = df_nubank[df_nubank['cvc_data_supplied'] == False]

In [7]:
# treatment
success_a = group_cvc['authorization'].sum()
size_a = len(group_cvc)

# control
success_b = group_no_cvc['authorization'].sum()
size_b = len(group_no_cvc)

In [8]:
# test result
stat, p_value = run_proportion_test(
    success_a,
    size_a,
    success_b,
    size_b,
    'larger'
)

print(f"\nStatistical test: {stat:.4f}")
print(f"p-value: {p_value:.10e}")


Statistical test: 17.9173
p-value: 4.3184440330e-72


In [9]:
# interpretation
alpha = 0.05

if p_value < alpha:
    print("Reject H0")
else:
    print("Fail to reject H0")

Reject H0


**Interpretation**
* As the p-value is below the significance threshold of 0.05, the null hypothesis is rejected.
* Transactions with CVC supplied have statistically significantly higher authorization rates than transactions without CVC supplied.

**Recommendation**
* The ecommerce should consider encouraging or enforcing CVC collection in applicable checkout flows to improve authorization performance from Nubank transactions.

## Saved Cards / ContAuth

**Business problem**

Recurring transactions are generally perceived as lower risk by issuers.

Note 1: this isn't a test on saving a card to get approval.

Note 2: ContAuth transactions are not randomly assigned and may already represent trusted customers, so the observed uplift should not be interpreted as purely causal.

<br>

**Analytical question**

Do recurring transactions (ContAuth) have higher authorization rates than Ecommerce transactions?

<br>

**Hypothesis**

H0: Recurring transactions have the same authorization rate as Ecommerce transactions.
* $H_0: p_{ContAuth} = p_{Ecommerce}$

H1: Recurring transactions have a higher authorization rate.
* $H_1: p_{ContAuth} > p_{Ecommerce}$

<br>

**Comparison**

| Group     | Definition             |
| :-------- | :--------------------- |
| Control   | Ecommerce transactions |
| Treatment | CountAuth transactions |


<br>

**Interpretation**

If p-value < 0.05:

* reject H0;
* statistical evidence that ContAth improves approval rates.

In [10]:
# create groups
group_contauth = df[
    df['shopper_interaction'] == 'ContAuth'
]

group_ecommerce = df[
    df['shopper_interaction'] == 'Ecommerce'
]

In [11]:
# treatment
success_a = group_contauth['authorization'].sum()
size_a = len(group_contauth)

# control
success_b = group_ecommerce['authorization'].sum()
size_b = len(group_ecommerce)

In [12]:
# test result
stat, p_value = run_proportion_test(
    success_a,
    size_a,
    success_b,
    size_b,
    'larger'
)

print(f"\nStatistical test: {stat:.4f}")
print(f"p-value: {p_value:.10e}")


Statistical test: 165.2392
p-value: 0.0000000000e+00


In [13]:
# interpretation
alpha = 0.05

if p_value < alpha:
    print("Reject H0")
else:
    print("Fail to reject H0")

Reject H0


**Interpretation**
* As the p-value is below the significance threshold of 0.05, the null hypothesis is rejected.
* Recurring transactions (`ContAuth`) have statistically significantly higher authorization rates than one-off Ecommerce transactions.
* This suggests that issuers may perceive recurring shoppers as lower risk, increasing approval likelihood.

**Recommendation**
* The ecommerce should consider expanding recurring payment and card-on-file strategies for returning customers.
* Additionally, reducing friction for trusted repeat shoppers may help increase overall authorization performance.

## Small / Test Amount

**Business problem**

Low-value transactions may be interpreted as card testing behavior and therefore receive lower authorization rates.

<br>

**Analytical question**

Do small amount transactions have lower authorization rates due to issuer fraud prevention behavior?

Definition of "small amount": `amount` <= 0.5

<br>

**Hypothesis**

H0: Smaller amounts have the same approval rate as larger ones.
* $H_0: p_{SmallAmount} = p_{RegularAmount}$

H1: Smaller amounts have a lower approval rate than larger amounts.
* $H_1: p_{SmallAmount} < p_{RegularAmount}$

<br>

**Comparison**

| Group     | Definition    |
| :-------- | :------------ |
| Control   | amount > 0.5  |
| Treatment | amount <= 0.5 |


<br>

**Interpretation**

If p-value < 0.05:

* reject H0;
* statistical evidence that small amount reduce approval rates.

In [14]:
# Create groups
df_test3 = df.copy()
df_test3['amount_group'] = df_test3['amount'].apply(
    lambda x: 'SmallAmount' if x <= 0.5 else 'RegularAmount'
)

# Aggregate metrics
summary_test3 = (
    df_test3
    .groupby('amount_group')
    .agg(
        transactions=('psp_reference', 'count'),
        approved=('authorization', 'sum'),
        approval_rate=('authorization', 'mean')
        )
    .reset_index()
)

# Summary
summary_test3['approval_rate'] = (
    summary_test3['approval_rate'] * 100
).round(2)

summary_test3


,amount_group,transactions,approved,approval_rate
0,RegularAmount,928441,736265,79.30
1,SmallAmount,4946,3951,79.88


In [15]:
# Hypothesis Test
# H0: p_small = p_regular
# H1: p_small < p_regular

# Successes
success = [
    summary_test3.loc[
        summary_test3['amount_group'] == 'SmallAmount',
        'approved'
    ].values[0],

    summary_test3.loc[
        summary_test3['amount_group'] == 'RegularAmount',
        'approved'
    ].values[0]]

# Sample sizes
nobs = [
    summary_test3.loc[
        summary_test3['amount_group'] == 'SmallAmount',
        'transactions'
    ].values[0],

    summary_test3.loc[
        summary_test3['amount_group'] == 'RegularAmount',
        'transactions'
    ].values[0]
]

# One-sided z-test
stat, p_value = proportions_ztest(
    count=success,
    nobs=nobs,
    alternative='smaller')

print(f"\nStatistical test: {stat:.4f}")
print(f"p-value: {p_value:.10e}")


Statistical test: 1.0068
p-value: 8.4298717906e-01


In [16]:
# interpretation
alpha = 0.05

if p_value < alpha:
    print("Reject H0")
else:
    print("Fail to reject H0")

Fail to reject H0


**Interpretation**

* As the p-value is above the significance threshold of 0.05, the null hypothesis cannot be rejected.
* There is no statistical evidence that small amount transactions reduce authorization rates.
* In fact, the observed result suggests that small amount transactions may have higher authorization rates than regular transactions.

Although low-value transactions are commonly associated with card testing attempts, the data does not support the hypothesis that they perform worse in authorization.

This may indicate that:
- issuer fraud systems are already effectively filtering malicious attempts before authorization;
- legitimate low-value purchases are common;
- or that transaction amount alone is not a strong fraud signal in this dataset.